# 🐾 Neural Image Classifier — Oxford-IIIT Pets
## Hyperparameter Experiment Log: 10-Model Comparison

This notebook trains a **SimpleCNN** (3 convolutional layers → global average pool → fully connected classifier) on the **Oxford-IIIT Pets dataset** (37 pet breed classes, ~7,400 training images). Ten separate training runs explore how learning rate, epoch count, dropout, and batch size affect final accuracy and loss.

**Architecture Summary:**
- Input: 128×128 RGB images
- Conv layers: 32 → 64 → 128 filters (3×3 kernels, ReLU, MaxPool2d)
- Global Average Pooling → FC(128→256) → Dropout → FC(256→37)
- Optimizer: Adam | Loss: CrossEntropyLoss

**Key Findings Preview:**

| Model | LR | Epochs | Dropout | Batch | Accuracy |
|-------|----|--------|---------|-------|----------|
| Baseline | 0.001 | 5 | 0.3 | 32 | 5.11% |
| Model 1 | 0.001 | 5 | 0.03 | 16 | 5.54% |
| Model 2 | 0.0003 | 5 | 0.3 | 32 | 4.89% |
| Model 3 | 0.001 | 30 | 0.3 | 32 | 21.77% |
| Model 4 | 0.001 | 5 | 0.6 | 32 | 4.08% |
| Model 5 | 0.001 | 5 | 0.3 | 64 | 5.00% |
| Model 6 | 0.0003 | 30 | 0.4 | 64 | 10.00% |
| Model 7 | 0.001 | 30 | 0.4 | 64 | 18.51% |
| Model 8 | 0.0001 | 10 | 0.4 | 64 | 3.61% |
| **Model 9** ⭐ | **0.001** | **60** | **0.04** | **16** | **56.20%** |


In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

print("Downloading Oxford Pets dataset (this may take a minute)...")
dataset = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",
    download=True,
    transform=transform
)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

print("Total images:", len(dataset))
print("Number of classes:", len(dataset.classes))

LEARNING_RATE = 1e-3
EPOCHS = 5
DROPOUT_RATE = 0.3
NUM_CLASSES = len(dataset.classes)


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(256, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


model = SimpleCNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print("\nStarting training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    accuracy = correct / total
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(loader):.4f} | Accuracy: {accuracy:.4f}")

save_dir = os.path.dirname(os.path.abspath(__file__))
save_path = os.path.join(save_dir, "simple_cnn_pets.pt")
torch.save(model.state_dict(), save_path)

print("Model saved to:", save_path)


## 📋 Baseline Model — Results Summary

**Hyperparameters:** LR = 0.001 | Epochs = 5 | Dropout = 0.3 | Batch Size = 32

**Final Results:** Loss = 3.5479 | Accuracy = **5.11%** | Training Time = 115.6s

### What Happened
The baseline establishes the starting point for all experiments. With only 5 training epochs the model barely scratched the surface of the Oxford-IIIT Pets dataset, which contains 37 fine-grained pet breeds — a notoriously difficult classification challenge. Loss dropped only slightly from 3.6152 to 3.5479 across the five epochs and accuracy never climbed above 5%, which is essentially near-random performance for a 37-class problem (random chance ≈ 2.7%).

### Hyperparameter Explanation
- **Learning Rate (0.001):** A standard starting point for Adam. It caused consistent but very slow loss reduction — the model was learning, just not given enough time.
- **Epochs (5):** Far too few for a 37-class image classification task from scratch. The model needed many more passes through the data to develop meaningful visual representations.
- **Dropout (0.3):** A moderate regularization rate — appropriate but inconsequential at this stage given the model had barely begun learning.
- **Batch Size (32):** A reliable default. Gradients are computed over 32 images at a time, providing a decent balance of noise and stability.

### Takeaway
The baseline confirmed the architecture works and trains without errors. The real signal is that **epochs are the primary bottleneck** — the model needs significantly more training time to learn anything meaningful.


In [ ]:
"""
Model 1 - LR=0.001, EPOCHS=5, DROPOUT=0.03, BATCH_SIZE=16
"""
import os
import json
import time
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

MODEL_NAME = "model_1"
LEARNING_RATE = 0.001
EPOCHS = 5
DROPOUT_RATE = 0.03
BATCH_SIZE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

print("Loading Oxford Pets dataset...")
dataset = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",
    download=True,
    transform=transform
)

NUM_CLASSES = len(dataset.classes)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Total images:", len(dataset))
print("Number of classes:", NUM_CLASSES)
print(f"Hyperparameters: LR={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}")


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(256, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

epoch_logs = []
results_lines = []
print("\nStarting training...")
start_time = time.time()
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    epoch_logs.append({"epoch": epoch + 1, "loss": round(avg_loss, 4), "accuracy": round(accuracy, 4)})
    line = f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.4f}"
    print(line)
    results_lines.append(line)

training_time_sec = round(time.time() - start_time, 1)
print(f"\nTraining completed in {training_time_sec}s")

save_dir = os.getcwd()
save_path = os.path.join(save_dir, f"simple_cnn_pets_{MODEL_NAME}.pt")
torch.save(model.state_dict(), save_path)
print("Model saved to:", save_path)

results_dir = os.path.join(save_dir, "results")
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, f"{MODEL_NAME}_results.txt"), "w") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write(f"LEARNING_RATE={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT_RATE={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}\n")
    f.write("\n".join(results_lines))

# Update training_results.json
results_json_path = os.path.join(save_dir, "training_results.json")
if os.path.exists(results_json_path):
    with open(results_json_path, "r") as f:
        all_results = json.load(f)
else:
    all_results = {}
all_results["Model 1"] = {
    "hyperparameters": {"lr": LEARNING_RATE, "epochs": EPOCHS, "dropout": DROPOUT_RATE, "batch_size": BATCH_SIZE},
    "final_loss": epoch_logs[-1]["loss"],
    "final_accuracy": epoch_logs[-1]["accuracy"],
    "training_time_sec": training_time_sec,
    "epoch_logs": epoch_logs,
    "model_file": f"simple_cnn_pets_{MODEL_NAME}.pt"
}
with open(results_json_path, "w") as f:
    json.dump(all_results, f, indent=2)
print("Results written to training_results.json")

## 📋 Model 1 — Results Summary

**Hyperparameters:** LR = 0.001 | Epochs = 5 | Dropout = 0.03 | Batch Size = 16

**Final Results:** Loss = 3.5219 | Accuracy = **5.54%** | Training Time = 126.1s

### What Happened
Model 1 keeps the same learning rate and epoch count as the baseline but uses lower dropout (0.03 vs 0.3) and a smaller batch size (16 vs 32). Final accuracy improved slightly to 5.54% from the baseline’s 5.11%, and loss dropped a bit more (3.5219 vs 3.5479). The milder dropout leaves more capacity active during training, and the smaller batches provide noisier gradients that can help in short runs.

### Hyperparameter Explanation
- **Learning Rate (0.001):** Same as baseline; still a reasonable choice for Adam.
- **Epochs (5):** Same as baseline — the main limit remains too few passes over the data.
- **Dropout (0.03):** Much lower than baseline (0.3), so the model is less regularized and can fit the training set a bit better in only 5 epochs.
- **Batch Size (16):** Smaller than baseline (32), giving more parameter updates per epoch and potentially slightly faster learning early on.

### Takeaway
Model 1 shows that reducing dropout and batch size can yield a small gain when training for only 5 epochs, but the gain is modest. Meaningful improvement still requires more epochs, as later models demonstrate.

In [ ]:
"""
Model 2 - Better LEARNING_RATE.
LEARNING_RATE = 3e-4, EPOCHS = 5, DROPOUT_RATE = 0.3, BATCH_SIZE = 32
"""
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

MODEL_NAME = "model_2"
LEARNING_RATE = 3e-4
EPOCHS = 5
DROPOUT_RATE = 0.3
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

print("Loading Oxford Pets dataset...")
dataset = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",
    download=True,
    transform=transform
)

NUM_CLASSES = len(dataset.classes)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Total images:", len(dataset))
print("Number of classes:", NUM_CLASSES)
print(f"Hyperparameters: LR={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}")


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(256, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

results_lines = []
print("\nStarting training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    accuracy = correct / total
    line = f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(loader):.4f} | Accuracy: {accuracy:.4f}"
    print(line)
    results_lines.append(line)

save_dir = os.path.dirname(os.path.abspath(__file__))
save_path = os.path.join(save_dir, f"simple_cnn_pets_{MODEL_NAME}.pt")
torch.save(model.state_dict(), save_path)
print("Model saved to:", save_path)

results_dir = os.path.join(save_dir, "results")
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, f"{MODEL_NAME}_results.txt"), "w") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write(f"LEARNING_RATE={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT_RATE={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}\n")
    f.write("\n".join(results_lines))


## 📋 Model 2 — Results Summary

**Hyperparameters:** LR = 0.0003 | Epochs = 5 | Dropout = 0.3 | Batch Size = 32

**Final Results:** Loss = 3.5567 | Accuracy = **4.89%** | Training Time = 115.9s

### What Happened
Model 2 tested whether lowering the learning rate from 0.001 to 0.0003 would help. It did not — at least not within 5 epochs. The final accuracy (4.89%) was actually *slightly worse* than the baseline (5.11%), and loss was marginally higher (3.5567 vs 3.5479). The difference is so small it's within noise, but the trend is clear: a lower learning rate with the same number of epochs simply moves slower without arriving anywhere better.

### Hyperparameter Explanation
- **Learning Rate (0.0003):** A more conservative step size causes smaller weight updates each batch. This can be beneficial in long training runs where precision matters, but over only 5 epochs it means the model traverses less of the loss landscape and learns less.
- **Epochs (5):** Same constraint as baseline — not enough iterations to compensate for the slower learning rate.
- **Dropout (0.3) / Batch Size (32):** Held constant from baseline — correctly isolating the learning rate as the only variable.

### Takeaway
**Lower learning rate ≠ better model** when training time is fixed. LR 0.0003 needs more epochs to outperform LR 0.001. This experiment ruled out reducing the learning rate as a quick fix and pointed toward increasing training duration instead.


In [ ]:
"""
Model 3 - Better EPOCHS.
LEARNING_RATE = 1e-3, EPOCHS = 30, DROPOUT_RATE = 0.3, BATCH_SIZE = 32
"""
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

MODEL_NAME = "model_3"
LEARNING_RATE = 1e-3
EPOCHS = 30
DROPOUT_RATE = 0.3
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

print("Loading Oxford Pets dataset...")
dataset = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",
    download=True,
    transform=transform
)

NUM_CLASSES = len(dataset.classes)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Total images:", len(dataset))
print("Number of classes:", NUM_CLASSES)
print(f"Hyperparameters: LR={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}")


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(256, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

results_lines = []
print("\nStarting training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    accuracy = correct / total
    line = f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(loader):.4f} | Accuracy: {accuracy:.4f}"
    print(line)
    results_lines.append(line)

save_dir = os.path.dirname(os.path.abspath(__file__))
save_path = os.path.join(save_dir, f"simple_cnn_pets_{MODEL_NAME}.pt")
torch.save(model.state_dict(), save_path)
print("Model saved to:", save_path)

results_dir = os.path.join(save_dir, "results")
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, f"{MODEL_NAME}_results.txt"), "w") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write(f"LEARNING_RATE={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT_RATE={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}\n")
    f.write("\n".join(results_lines))


## 📋 Model 3 — Results Summary

**Hyperparameters:** LR = 0.001 | Epochs = 30 | Dropout = 0.3 | Batch Size = 32

**Final Results:** Loss = 2.7415 | Accuracy = **21.77%** | Training Time = 724.7s

### What Happened
This was the first major breakthrough. By simply running the same baseline configuration for 30 epochs instead of 5, accuracy jumped from 5.11% to 21.77% — a **4× improvement**. Loss fell from the baseline's 3.5479 all the way to 2.7415. The training curve shows steady, consistent improvement across all 30 epochs with no plateau, which indicates the model had not yet reached its learning ceiling.

### Hyperparameter Explanation
- **Learning Rate (0.001):** Kept from baseline. Proved effective at this higher epoch count — the model now had enough steps to meaningfully navigate the loss landscape.
- **Epochs (30):** The defining change. The model had 6× more passes through the data, allowing the convolutional layers to develop genuinely useful visual features like ears, fur patterns, and snout shapes across the 37 breeds.
- **Dropout (0.3):** Kept constant for a clean comparison. Continued to provide mild regularization during the longer run.
- **Batch Size (32):** Unchanged from baseline.

### Takeaway
**More epochs is the single biggest lever** for this model and dataset. The architecture is capable — it just needs time. This result became the reference point for all subsequent multi-epoch experiments.


In [ ]:
"""
Model 4 - Higher DROPOUT_RATE.
LEARNING_RATE = 1e-3, EPOCHS = 5, DROPOUT_RATE = 0.6, BATCH_SIZE = 32
"""
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

MODEL_NAME = "model_4"
LEARNING_RATE = 1e-3
EPOCHS = 5
DROPOUT_RATE = 0.6
BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

print("Loading Oxford Pets dataset...")
dataset = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",
    download=True,
    transform=transform
)

NUM_CLASSES = len(dataset.classes)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Total images:", len(dataset))
print("Number of classes:", NUM_CLASSES)
print(f"Hyperparameters: LR={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}")


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(256, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

results_lines = []
print("\nStarting training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    accuracy = correct / total
    line = f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(loader):.4f} | Accuracy: {accuracy:.4f}"
    print(line)
    results_lines.append(line)

save_dir = os.path.dirname(os.path.abspath(__file__))
save_path = os.path.join(save_dir, f"simple_cnn_pets_{MODEL_NAME}.pt")
torch.save(model.state_dict(), save_path)
print("Model saved to:", save_path)

results_dir = os.path.join(save_dir, "results")
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, f"{MODEL_NAME}_results.txt"), "w") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write(f"LEARNING_RATE={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT_RATE={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}\n")
    f.write("\n".join(results_lines))


## 📋 Model 4 — Results Summary

**Hyperparameters:** LR = 0.001 | Epochs = 5 | Dropout = 0.6 | Batch Size = 32

**Final Results:** Loss = 3.5758 | Accuracy = **4.08%** | Training Time = 120.8s

### What Happened
Model 4 tested whether increasing dropout regularization would improve short-run performance. The result was the opposite — accuracy *decreased* to 4.08%, the worst 5-epoch result of all models tested. Higher dropout forced the model to randomly zero out 60% of activations during each forward pass, which severely restricted learning capacity over the already-short training window.

### Hyperparameter Explanation
- **Dropout (0.6):** Double the baseline rate. Dropout is a regularization technique designed to prevent overfitting by forcing the network to learn redundant representations. However, at 0.6 with only 5 epochs, the model is spending more time "forgetting" than learning. This is especially harmful early in training when the network hasn't yet converged.
- **Learning Rate (0.001):** Unchanged. Not enough to overcome the handicap imposed by high dropout.
- **Epochs (5):** Too few — high dropout requires more training time to compensate for the reduced learning signal per step.
- **Batch Size (32):** Unchanged.

### Takeaway
**High dropout without sufficient training epochs actively hurts performance.** Dropout is best applied either moderately or in longer training runs where the model has time to develop robust features despite random deactivation. For this architecture, 0.3 is the better setting.


In [ ]:
"""
Model 5 - Higher BATCH_SIZE.
LEARNING_RATE = 1e-3, EPOCHS = 5, DROPOUT_RATE = 0.3, BATCH_SIZE = 64
"""
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

MODEL_NAME = "model_5"
LEARNING_RATE = 1e-3
EPOCHS = 5
DROPOUT_RATE = 0.3
BATCH_SIZE = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

print("Loading Oxford Pets dataset...")
dataset = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",
    download=True,
    transform=transform
)

NUM_CLASSES = len(dataset.classes)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Total images:", len(dataset))
print("Number of classes:", NUM_CLASSES)
print(f"Hyperparameters: LR={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}")


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(256, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

results_lines = []
print("\nStarting training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    accuracy = correct / total
    line = f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(loader):.4f} | Accuracy: {accuracy:.4f}"
    print(line)
    results_lines.append(line)

save_dir = os.path.dirname(os.path.abspath(__file__))
save_path = os.path.join(save_dir, f"simple_cnn_pets_{MODEL_NAME}.pt")
torch.save(model.state_dict(), save_path)
print("Model saved to:", save_path)

results_dir = os.path.join(save_dir, "results")
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, f"{MODEL_NAME}_results.txt"), "w") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write(f"LEARNING_RATE={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT_RATE={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}\n")
    f.write("\n".join(results_lines))


## 📋 Model 5 — Results Summary

**Hyperparameters:** LR = 0.001 | Epochs = 5 | Dropout = 0.3 | Batch Size = 64

**Final Results:** Loss = 3.5554 | Accuracy = **5.00%** | Training Time = 117.0s

### What Happened
Model 5 doubled the batch size from 32 to 64 while keeping everything else at baseline settings. The result was nearly identical to the baseline — a marginal improvement in loss (3.5554 vs 3.5479) with essentially the same accuracy (~5%). The larger batch produced smoother gradient estimates but provided no meaningful improvement in 5 epochs.

### Hyperparameter Explanation
- **Batch Size (64):** Larger batches compute gradients over more samples simultaneously, producing lower-variance (smoother) gradient estimates. This can speed up wall-clock training since fewer steps are taken per epoch — but each step is more computationally expensive and the model sees the same data overall.
- **Training Time (~117s):** Nearly identical to baseline, showing the larger batch didn't dramatically change compute cost in this setup.
- **LR/Epochs/Dropout:** All held at baseline values to isolate the batch size variable.

### Takeaway
**Batch size alone has minimal impact within 5 epochs.** Larger batches are often paired with higher learning rates (linear scaling rule) to take advantage of smoother gradients. Without that adjustment, Model 5 behaved almost identically to the baseline.


In [ ]:
"""
Model 6 - Combined: LR=3e-4, EPOCHS=30, DROPOUT=0.4, BATCH_SIZE=64
"""
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

MODEL_NAME = "model_6"
LEARNING_RATE = 3e-4
EPOCHS = 30
DROPOUT_RATE = 0.4
BATCH_SIZE = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

print("Loading Oxford Pets dataset...")
dataset = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",
    download=True,
    transform=transform
)

NUM_CLASSES = len(dataset.classes)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Total images:", len(dataset))
print("Number of classes:", NUM_CLASSES)
print(f"Hyperparameters: LR={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}")


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(256, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

results_lines = []
print("\nStarting training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    accuracy = correct / total
    line = f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(loader):.4f} | Accuracy: {accuracy:.4f}"
    print(line)
    results_lines.append(line)

save_dir = os.path.dirname(os.path.abspath(__file__))
save_path = os.path.join(save_dir, f"simple_cnn_pets_{MODEL_NAME}.pt")
torch.save(model.state_dict(), save_path)
print("Model saved to:", save_path)

results_dir = os.path.join(save_dir, "results")
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, f"{MODEL_NAME}_results.txt"), "w") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write(f"LEARNING_RATE={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT_RATE={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}\n")
    f.write("\n".join(results_lines))


## 📋 Model 6 — Results Summary

**Hyperparameters:** LR = 0.0003 | Epochs = 30 | Dropout = 0.4 | Batch Size = 64

**Final Results:** Loss = 3.3082 | Accuracy = **10.00%** | Training Time = 668.4s

### What Happened
Model 6 combined three changes at once — lower LR, higher dropout, and larger batch — while running for 30 epochs. It improved substantially over any 5-epoch run, but fell far short of Model 3's 21.77% accuracy at the same epoch count. The model showed slow, steady gains but the lower learning rate (0.0003) meaningfully dragged down the learning speed. By epoch 30, accuracy had only reached 10% — less than half of Model 3's result.

### Hyperparameter Explanation
- **LR (0.0003):** The primary culprit for underperformance. Even over 30 epochs, this LR moved too cautiously. Smaller steps meant the model made less progress per pass through the data.
- **Epochs (30):** Adequate duration, but could not fully compensate for the slow LR.
- **Dropout (0.4):** Slightly higher than baseline. Added more regularization pressure which, combined with a slow LR, restricted how much the model could learn.
- **Batch Size (64):** Fewer gradient update steps per epoch compared to batch size 32, compounding the already-slow learning pace of the lower LR.

### Takeaway
**Combining multiple conservative changes compounded their downsides.** Lower LR + higher dropout + larger batch all individually slow learning — stacking them without compensating elsewhere (e.g., many more epochs) yielded a model that learned at roughly half the rate of Model 3. This highlights the importance of understanding interaction effects between hyperparameters.


In [ ]:

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(256, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

results_lines = []
print("\nStarting training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    accuracy = correct / total
    line = f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(loader):.4f} | Accuracy: {accuracy:.4f}"
    print(line)
    results_lines.append(line)

save_dir = os.path.dirname(os.path.abspath(__file__))
save_path = os.path.join(save_dir, f"simple_cnn_pets_{MODEL_NAME}.pt")
torch.save(model.state_dict(), save_path)
print("Model saved to:", save_path)

results_dir = os.path.join(save_dir, "results")
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, f"{MODEL_NAME}_results.txt"), "w") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write(f"LEARNING_RATE={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT_RATE={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}\n")
    f.write("\n".join(results_lines))


## 📋 Model 7 — Results Summary

**Hyperparameters:** LR = 0.001 | Epochs = 30 | Dropout = 0.4 | Batch Size = 64

**Final Results:** Loss = 2.8989 | Accuracy = **18.51%** | Training Time = 798.6s

### What Happened
Model 7 kept the LR from the best-performing experiment so far (Model 3) while testing larger batch size and slightly higher dropout over 30 epochs. It performed well but fell short of Model 3's 21.77%, landing at 18.51%. The training curve showed rapid acceleration around epochs 15–20 before stabilizing, suggesting the model was still capable of learning but the larger batch and higher dropout slowed the pace of improvement.

### Hyperparameter Explanation
- **LR (0.001):** Correctly maintained from Model 3. This rate proved effective at the 30-epoch scale.
- **Epochs (30):** Same count as Model 3 for a direct comparison. The model was still improving at epoch 30 — it hadn't plateaued — suggesting it could have benefited from additional training.
- **Dropout (0.4):** Slightly higher than Model 3's 0.3. Added enough regularization to slightly limit learning throughput, likely responsible for some of the 3-point accuracy gap vs. Model 3.
- **Batch Size (64):** Fewer weight updates per epoch vs. batch 32. Over 30 epochs, this means the model made fewer total gradient steps, translating to slightly less learned.

### Takeaway
**LR 0.001 remains the key ingredient** for high performance at 30 epochs. The larger batch and marginally higher dropout mildly reduced final accuracy. Model 7 confirmed that Model 3's hyperparameter combination was close to optimal for this training duration.


In [ ]:
"""
Model 8 (second "Model 7" in spec) - LR=3e-4, EPOCHS=10, DROPOUT=0.4, BATCH_SIZE=64
"""
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

MODEL_NAME = "model_8"
LEARNING_RATE = 1e-4
EPOCHS = 10
DROPOUT_RATE = 0.4
BATCH_SIZE = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

print("Loading Oxford Pets dataset...")
dataset = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",
    download=True,
    transform=transform
)

NUM_CLASSES = len(dataset.classes)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Total images:", len(dataset))
print("Number of classes:", NUM_CLASSES)
print(f"Hyperparameters: LR={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}")


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(256, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

results_lines = []
print("\nStarting training...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    accuracy = correct / total
    line = f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss/len(loader):.4f} | Accuracy: {accuracy:.4f}"
    print(line)
    results_lines.append(line)

save_dir = os.path.dirname(os.path.abspath(__file__))
save_path = os.path.join(save_dir, f"simple_cnn_pets_{MODEL_NAME}.pt")
torch.save(model.state_dict(), save_path)
print("Model saved to:", save_path)

results_dir = os.path.join(save_dir, "results")
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, f"{MODEL_NAME}_results.txt"), "w") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write(f"LEARNING_RATE={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT_RATE={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}\n")
    f.write("\n".join(results_lines))


## 📋 Model 8 — Results Summary

**Hyperparameters:** LR = 0.0001 | Epochs = 10 | Dropout = 0.4 | Batch Size = 64

**Final Results:** Loss = 3.5836 | Accuracy = **3.61%** | Training Time = N/A

### What Happened
Model 8 was the worst-performing model in the entire experiment series. With a very low learning rate (0.0001), only 10 epochs, and larger batch size, the model barely moved from its random initialization. Loss barely decreased (3.6126 → 3.5836) and accuracy peaked at just 3.61% — essentially random guessing for a 37-class problem. Every epoch log shows the model inching forward at an almost imperceptible rate.

### Hyperparameter Explanation
- **LR (0.0001):** Ten times smaller than the baseline. This is far too conservative — gradient updates were so small that the model's weights could barely shift from their initialized values. Even over 10 epochs, the cumulative learning was negligible.
- **Epochs (10):** Not nearly enough to compensate for the near-zero LR. The model needed at least 100+ epochs at this learning rate to show meaningful progress.
- **Dropout (0.4):** Added regularization on top of a model that hadn't yet learned anything worth regularizing.
- **Batch Size (64):** Combined with the low LR, this further limited the number of useful gradient updates per epoch.

### Takeaway
**A learning rate of 0.0001 is far too low for this architecture and dataset.** This model demonstrates a failure mode: when the LR is set too small, training effectively stalls regardless of other settings. It serves as a useful lower bound — any LR below ~0.0003 appears to be counterproductive within a reasonable epoch budget.


In [ ]:
"""
Model 9 - LR=1e-3, EPOCHS=60, DROPOUT=0.04, BATCH_SIZE=16
"""
import os
import json
import time
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

MODEL_NAME = "model_9"
LEARNING_RATE = 1e-3
EPOCHS = 60
DROPOUT_RATE = 0.04
BATCH_SIZE = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

print("Loading Oxford Pets dataset...")
dataset = datasets.OxfordIIITPet(
    root="./data",
    split="trainval",
    download=True,
    transform=transform
)

NUM_CLASSES = len(dataset.classes)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Total images:", len(dataset))
print("Number of classes:", NUM_CLASSES)
print(f"Hyperparameters: LR={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}")


class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(256, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.global_pool(x)
        x = self.classifier(x)
        return x


model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

epoch_logs = []
results_lines = []
print("\nStarting training...")
start_time = time.time()
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    epoch_logs.append({"epoch": epoch + 1, "loss": round(avg_loss, 4), "accuracy": round(accuracy, 4)})
    line = f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.4f}"
    print(line)
    results_lines.append(line)

training_time_sec = round(time.time() - start_time, 1)
print(f"\nTraining completed in {training_time_sec}s")

save_dir = os.path.dirname(os.path.abspath(__file__))
save_path = os.path.join(save_dir, f"simple_cnn_pets_{MODEL_NAME}.pt")
torch.save(model.state_dict(), save_path)
print("Model saved to:", save_path)

results_dir = os.path.join(save_dir, "results")
os.makedirs(results_dir, exist_ok=True)
with open(os.path.join(results_dir, f"{MODEL_NAME}_results.txt"), "w") as f:
    f.write(f"MODEL: {MODEL_NAME}\n")
    f.write(f"LEARNING_RATE={LEARNING_RATE}, EPOCHS={EPOCHS}, DROPOUT_RATE={DROPOUT_RATE}, BATCH_SIZE={BATCH_SIZE}\n")
    f.write("\n".join(results_lines))

# Update training_results.json
results_json_path = os.path.join(save_dir, "training_results.json")
if os.path.exists(results_json_path):
    with open(results_json_path, "r") as f:
        all_results = json.load(f)
else:
    all_results = {}
all_results["Model 9"] = {
    "hyperparameters": {
        "lr": LEARNING_RATE,
        "epochs": EPOCHS,
        "dropout": DROPOUT_RATE,
        "batch_size": BATCH_SIZE
    },
    "final_loss": epoch_logs[-1]["loss"],
    "final_accuracy": epoch_logs[-1]["accuracy"],
    "training_time_sec": training_time_sec,
    "epoch_logs": epoch_logs,
    "model_file": f"simple_cnn_pets_{MODEL_NAME}.pt"
}
with open(results_json_path, "w") as f:
    json.dump(all_results, f, indent=2)
print("Results written to training_results.json")


## 🏆 Model 9 — Results Summary (Best Model)

**Hyperparameters:** LR = 0.001 | Epochs = 60 | Dropout = 0.04 | Batch Size = 16

**Final Results:** Loss = 1.4251 | Accuracy = **56.20%** | Training Time = N/A

### What Happened
Model 9 is the standout winner by a massive margin. It achieved 56.20% accuracy — **more than 2.5× better than Model 3 (21.77%)** and nearly 11× better than the baseline. The training curve shows consistent, uninterrupted improvement across all 60 epochs with no sign of plateau, suggesting performance could potentially improve further with additional training. Loss dropped dramatically from 3.6155 to 1.4251 — by far the largest absolute loss reduction of any experiment.

### Hyperparameter Explanation
- **Learning Rate (0.001):** Confirmed again as the optimal rate for this architecture. Combined with 60 epochs it allowed the model to navigate the loss landscape thoroughly.
- **Epochs (60):** The single biggest driver of success. Double the previous best (Model 3's 30 epochs) gave the model 60 full passes through the dataset, allowing it to refine visual feature representations across all 37 pet breeds in fine-grained detail.
- **Dropout (0.04):** The key differentiator. Near-zero dropout allowed the model to retain almost all learned activations during training. With 60 epochs providing plenty of capacity, overfitting was less of an immediate concern, and the extremely low dropout enabled maximum information flow through the network at every step.
- **Batch Size (16):** Smaller batches mean more gradient updates per epoch (roughly 2× vs. batch 32, 4× vs. batch 64). This created a noisier but more frequent learning signal, helping the model escape local minima and continuously refine its weights. The combination of small batches + 60 epochs effectively multiplied the total number of weight updates dramatically compared to all previous runs.

### Why This Combination Won
The three levers pulled in the right direction simultaneously:
1. **More epochs** → more total learning time
2. **Lower dropout** → maximum learning capacity per step
3. **Smaller batch** → more gradient updates per epoch, faster weight refinement

### Takeaway
For this SimpleCNN architecture on Oxford-IIIT Pets: **train longer, drop less, batch smaller.** Model 9's result demonstrates that the architecture has substantially more capacity than earlier experiments revealed — it was simply undertrained and over-regularized. Future improvements should explore even more epochs, learning rate scheduling (e.g., cosine annealing), and data augmentation to push accuracy higher without overfitting.
